In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# LOAD DATA

In [ ]:
import pandas as pd

df = pd.read_csv("/content/song_recomendation_A.csv")
df.head()

In [ ]:
df.shape[0] # jumlah baris

In [ ]:
df.shape[1] # jumlah kolom

In [ ]:
df.info()

# EDA

## missing value

In [ ]:
missing_data = df.isnull().sum()
missing_percentage = (missing_data / len(df)) * 100
print(pd.concat([missing_data, missing_percentage], axis=1, keys=['Total Missing Value', 'Percent']))

### numerical checks

In [ ]:
numerical_cols = df.select_dtypes(include=np.number).columns
df[numerical_cols].skew()

In [ ]:
# drop missing value dari 'Track Name', 'Artist Name(s)', 'Album Name' karena hanya 1
df = df.dropna(subset=['Track Name', 'Artist Name(s)', 'Album Name'])

In [ ]:
df['Acousticness'].describe()

In [ ]:
# impute acousticness menggunakan median karena right-skewed

median_acoustic = df['Acousticness'].median()
df['Acousticness'] = df['Acousticness'].fillna(median_acoustic)

In [ ]:
df['Tempo'].describe()

In [ ]:
# impute tempo menggunakan median karena slighty right_skewed (cari aman menggunakan median)
median_tempo = df['Tempo'].median()
df['Tempo'] = df['Tempo'].fillna(median_tempo)

### categorical checks

In [ ]:
cat_cols = df.select_dtypes(include='object').columns
cat_cols

for col in cat_cols:
    print(f"\n=== {col} ===")
    print(df[col].value_counts(dropna=False))

In [ ]:
# mengganti kolom 'Artist Genres' dengan unknown
df['Artist Genres'] = df['Artist Genres'].fillna('Unknown')

## duplicate

In [ ]:
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates()
df.duplicated().sum()

## outliers

In [ ]:
df.describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

num_cols = [
    'Danceability', 'Energy', 'Loudness', 'Speechiness',
    'Acousticness', 'Instrumentalness', 'Liveness',
    'Valence', 'Tempo'
]

for col in num_cols:
    plt.figure(figsize=(6, 2))
    sns.boxplot(x=df[col])
    plt.title(f'Boxplot of {col}')
    plt.xlabel(col)
    plt.show()


Based on boxplot analysis, several statistical outliers were identified across multiple audio features. However, these values were retained as they represent valid musical characteristics rather than data errors.

In [ ]:
df.info()

In [ ]:
# drop kolom irrelevant
cols_to_drop = [
    'Track URI',
    'Album Name',
    'Album Release Date',
    'Artist Genres',
    'Key',
    'Time Signature'
]

df_clean = df.drop(columns=cols_to_drop)


In [ ]:
feature_cols = [
    'Danceability',
    'Energy',
    'Loudness',
    'Speechiness',
    'Acousticness',
    'Instrumentalness',
    'Liveness',
    'Valence',
    'Tempo',
    'Mode'
]

X = df_clean[feature_cols]

In [ ]:
metadata = df_clean[[
    'Track Name',
    'Artist Name(s)',
    'Popularity'
]]

# PENJELASAN RECOMMENDATION SYSTEM

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

similarity_matrix = cosine_similarity(X_scaled)

In [ ]:
def recommend_songs(song_title, df, similarity_matrix, top_n=5):
    """
    Input  : song_title (str)
    Output : DataFrame berisi 5 lagu rekomendasi
    """

    # Cek apakah lagu ada di dataset
    if song_title not in df['Track Name'].values:
        return "Song not found in the dataset."

    # index lagu input
    idx = df[df['Track Name'] == song_title].index[0]

    # Ambil similarity score
    similarity_scores = list(enumerate(similarity_matrix[idx]))

    # Urutkan berdasarkan similarity tertinggi
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    # Ambil top-N (skip lagu itu sendiri)
    top_indices = [i[0] for i in similarity_scores[1:top_n+1]]

    # Return hasil rekomendasi
    return df.loc[top_indices, ['Track Name', 'Artist Name(s)', 'Popularity']]


In [ ]:
input_song = input("Input song: ")

if input_song not in df_clean['Track Name'].values:
    print("Song not found in the dataset.")
else:
    recommendations = recommend_songs(
        input_song,
        df_clean,
        similarity_matrix
    )
    print("\nRecommended Songs:")
    display(recommendations)


## Evaluation

### song 1

In [ ]:
recommend_songs("All About That Bass", df_clean, similarity_matrix)

In [ ]:
input_song = "All About That Bass"

input_features = df_clean[
    df_clean['Track Name'] == input_song
][['Track Name', 'Artist Name(s)'] + feature_cols]

recommended_titles = recommend_songs(
    input_song,
    df_clean,
    similarity_matrix
)['Track Name'].tolist()

recommended_songs = df_clean[
    df_clean['Track Name'].isin(recommended_titles)
][['Track Name', 'Artist Name(s)'] + feature_cols]

final_feature_table = pd.concat(
    [input_features, recommended_songs],
    ignore_index=True
)

final_feature_table


In [ ]:
input_features = df_clean[
    df_clean['Track Name'] == "All About That Bass"
][feature_cols]

input_features

recommended_titles = recommend_songs(
    "All About That Bass",
    df_clean,
    similarity_matrix
)['Track Name'].tolist()

recommended_songs = df_clean[
    df_clean['Track Name'].isin(recommended_titles)
]

recommended_features = recommended_songs[feature_cols]

recommended_features

comparison = pd.DataFrame({
    'Input Song (All About That Bass)': input_features.mean(),
    'Recommended Songs (Average)': recommended_features.mean()
})

comparison

### song 2

In [ ]:
recommend_songs("Begin Again", df_clean, similarity_matrix)

In [ ]:
input_song = "Begin Again"

input_features = df_clean[
    df_clean['Track Name'] == input_song
][['Track Name', 'Artist Name(s)'] + feature_cols]

recommended_titles = recommend_songs(
    input_song,
    df_clean,
    similarity_matrix
)['Track Name'].tolist()

recommended_songs = df_clean[
    df_clean['Track Name'].isin(recommended_titles)
][['Track Name', 'Artist Name(s)'] + feature_cols]

final_feature_table = pd.concat(
    [input_features, recommended_songs],
    ignore_index=True
)

final_feature_table

In [ ]:
input_features = df_clean[
    df_clean['Track Name'] == "Begin Again"
][feature_cols]

input_features

recommended_titles = recommend_songs(
    "Begin Again",
    df_clean,
    similarity_matrix
)['Track Name'].tolist()

recommended_songs = df_clean[
    df_clean['Track Name'].isin(recommended_titles)
]

recommended_features = recommended_songs[feature_cols]

recommended_features

comparison = pd.DataFrame({
    'Input Song (Begin Again)': input_features.mean(),
    'Recommended Songs (Average)': recommended_features.mean()
})

comparison


The recommendation results for “Begin Again” show strong alignment in danceability, energy, loudness, acousticness, and vocal presence. Although differences are observed in tempo and valence, the overall audio profile of the recommended songs remains consistent with the input track, demonstrating the effectiveness of the content-based recommendation system.

### song 3

In [ ]:
recommend_songs("It Will Rain", df_clean, similarity_matrix)

In [ ]:
input_song = "It Will Rain"

input_features = df_clean[
    df_clean['Track Name'] == input_song
][['Track Name', 'Artist Name(s)'] + feature_cols]

recommended_titles = recommend_songs(
    input_song,
    df_clean,
    similarity_matrix
)['Track Name'].tolist()

recommended_songs = df_clean[
    df_clean['Track Name'].isin(recommended_titles)
][['Track Name', 'Artist Name(s)'] + feature_cols]

final_feature_table = pd.concat(
    [input_features, recommended_songs],
    ignore_index=True
)

final_feature_table

In [ ]:
input_features = df_clean[
    df_clean['Track Name'] == "It Will Rain"
][feature_cols]

recommended_titles = recommend_songs(
    "It Will Rain",
    df_clean,
    similarity_matrix
)['Track Name'].tolist()

recommended_songs = df_clean[
    df_clean['Track Name'].isin(recommended_titles)
]

recommended_features = recommended_songs[feature_cols]

recommended_features

comparison = pd.DataFrame({
    'Input Song (It Will Rain)': input_features.mean(),
    'Recommended Songs (Average)': recommended_features.mean()
})

comparison



# RECOMMENDATION SYSTEM

In [267]:
while True:
    input_song = input("\nInput song: ")
    if input_song not in df_clean['Track Name'].values:
        print("Song not found in the dataset. Please try again.")
        continue

    recommendations = recommend_songs(
        input_song,
        df_clean,
        similarity_matrix
    )

    print("\nRecommended Songs:")
    display(recommendations)


    print("\nDo you want to search another song?")
    print("Type 1 to continue")
    print("Type 2 to exit")

    choice = input("Your choice: ")

    if choice == "2":
        print("\nThank you for using the recommendation system!")
        break
    elif choice == "1":
        continue
    else:
        print("\nInvalid input. The system will exit.")
        break



Input song: Style

Recommended Songs:


,Track Name,Artist Name(s),Popularity
2410,Light It Up (feat. Static Revenger),"Stan Walker, Static Revenger",26
2162,All Of The Dreamers,Powderfinger,23
4150,What You Need,INXS,0
2205,Help Me Make It Through The Night,Sammi Smith,58
2891,Angel,Amanda Perez,50



Do you want to search another song?
Type 1 to continue
Type 2 to exit
Your choice: 1

Input song: It Will Rain

Recommended Songs:


,Track Name,Artist Name(s),Popularity
1073,Do It Right,Anne-Marie,49
4943,Don't Hang Up,The Orlons,34
1500,I Can Help,Billy Swan,52
3778,Glad All Over - 2019 - Remaster,The Dave Clark Five,49
2500,The Power Of Love - Radio Edit,Céline Dion,0



Do you want to search another song?
Type 1 to continue
Type 2 to exit
Your choice: Clean

Invalid input. The system will exit.
